In [1]:
import pandas as pd
import numpy as np
from yahooquery import Ticker
from sklearn.preprocessing import MinMaxScaler
import json
from data_loader import get_or_fetch_historical_prices

def robust_scale(series, is_reverse=False):
    """自定義正規化：包含 1%~95% 縮尾處理，並壓縮至 0~1 區間"""
    lower_bound = series.quantile(0.01)
    upper_bound = series.quantile(0.95)
    clipped = series.clip(lower=lower_bound, upper=upper_bound)
    
    s_min = clipped.min()
    s_max = clipped.max()
    
    if s_max == s_min:
        scaled = np.zeros(len(clipped))
    else:
        scaled = (clipped - s_min) / (s_max - s_min)
        
    # 如果是風險或成本 (越小越好)，計算偏好分數時必須反向 (越高分代表效用越大)
    if is_reverse:
        return 1.0 - scaled
    return scaled

def run_stage2_5_preference_deduplication_yq():
    print("啟動 Stage 2.5: 偏好驅動去重與分群 (白名單過濾與原始數據重構)...")
    
    # 1. 讀取 Stage 2 的兩層級 9 維全局權重
    try:
        with open("json\\stage2_ahp_global_weights.json", "r", encoding="utf-8") as f:
            ahp_data = json.load(f)
            global_weights = ahp_data["Global_Weights"]
    except FileNotFoundError:
        print("❌ 找不到 stage2_ahp_global_weights.json。")
        return

    # 2. 讀取 Stage 1 白名單，與 Stage 0 原始數據
    try:
        df_candidates = pd.read_csv("csv\\stage1_final_candidates.csv")
        df_raw = pd.read_csv("csv\\stage0_final_matrix.csv")
    except FileNotFoundError:
        print("❌ 找不到 stage1 或 stage0 的 csv 檔案。")
        return

    # 使用 Stage 1 產出的 ETF 名單作為白名單過濾 Stage 0 的原始數據
    valid_tickers = df_candidates['ETF'].tolist()
    df = df_raw[df_raw['ETF'].isin(valid_tickers)].reset_index(drop=True)
    
    print(f"📥 成功載入 {len(df)} 檔候選 ETF 之原始特徵。")

    # 3. 9 大子特徵獨立正規化 (全面貫徹 1%~95% 縮尾處理)
    df_scaled = pd.DataFrame({'ETF': df['ETF']})
    
    # [正向特徵] 全面縮尾處理
    df_scaled['Norm_Return_CAGR'] = robust_scale(df['Return_CAGR (%)'])
    df_scaled['Norm_Return_Div'] = robust_scale(df['Return_Div (%)'])
    df_scaled['Norm_Div_Score'] = robust_scale(df['Div_Score (產出)'])
    df_scaled['Norm_FinBERT'] = robust_scale(df['FinBERT_score'])
    
    # [流動性特徵] 先取對數，再嚴格進行縮尾處理
    df_scaled['Norm_Liq_Volume'] = robust_scale(np.log1p(df['Liq_Volume (M)']))
    df_scaled['Norm_Liq_AUM'] = robust_scale(np.log1p(df['Liq_AUM (B)']))
    
    # [反向特徵] 縮尾處理後反轉 (1 - scaled)
    df_scaled['Norm_Risk_Vol'] = robust_scale(df['Risk_Vol (%)'], is_reverse=True)
    df_scaled['Norm_Risk_MaxDD'] = robust_scale(df['Risk_MaxDD (%)'].abs(), is_reverse=True)
    df_scaled['Norm_Cost_ExpRatio'] = robust_scale(df['Cost_ExpRatio (%)'], is_reverse=True)
    
    # 4. 計算使用者偏好分數 (User_Pref_Score)
    feature_map = {
        "Return_CAGR": 'Norm_Return_CAGR',
        "Return_Div": 'Norm_Return_Div',
        "Risk_Vol": 'Norm_Risk_Vol',
        "Risk_MaxDD": 'Norm_Risk_MaxDD',
        "Liq_Volume": 'Norm_Liq_Volume',
        "Liq_AUM": 'Norm_Liq_AUM',
        "Cost_ExpRatio": 'Norm_Cost_ExpRatio',
        "Div_Score": 'Norm_Div_Score',
        "FinBERT_score": 'Norm_FinBERT'
    }
    
    pref_scores = np.zeros(len(df))
    for key, weight in global_weights.items():
        if key in feature_map:
            col_name = feature_map[key]
            pref_scores += df_scaled[col_name].values * weight
        
    df['User_Pref_Score'] = pref_scores
    
    # 5. 下載歷史價格進行相關性計算
    price_matrix = get_or_fetch_historical_prices(valid_tickers)
    returns_matrix = price_matrix.pct_change().dropna(how='all')
    returns_matrix = returns_matrix.ffill().dropna(axis=1)
    
    # 對齊成功抓到價格的 ETF
    final_tickers = [t for t in valid_tickers if t in returns_matrix.columns]
    df = df[df['ETF'].isin(final_tickers)].reset_index(drop=True)
    corr_matrix = returns_matrix[final_tickers].corr()
    
    # 6. 相關性分群去重 (Threshold = 0.99)
    CORR_THRESHOLD = 0.99
    clusters = []
    processed_tickers = set()
    
    sorted_tickers = df.sort_values(by='User_Pref_Score', ascending=False)['ETF'].tolist()
    
    for ticker in sorted_tickers:
        if ticker in processed_tickers:
            continue
            
        correlated = corr_matrix.index[corr_matrix[ticker] >= CORR_THRESHOLD].tolist()
        cluster = [t for t in correlated if t not in processed_tickers]
        
        if cluster:
            clusters.append(cluster)
            processed_tickers.update(cluster)
            
    # 7. 挑選偏好分數最高代表
    final_portfolio_candidates = []
    print("\n=== 🎯 Stage 2.5 群集去重結果 ===")
    for i, cluster in enumerate(clusters):
        cluster_df = df[df['ETF'].isin(cluster)]
        best_etf = cluster_df.loc[cluster_df['User_Pref_Score'].idxmax()]
        final_portfolio_candidates.append(best_etf)
        
        if len(cluster) > 1:
            print(f"  > 群集 {cluster} -> 🏆 勝出: {best_etf['ETF']} (Score: {best_etf['User_Pref_Score']:.4f})")
            print(cluster_df[['ETF', 'User_Pref_Score']].to_string(index=False))
        
    final_df = pd.DataFrame(final_portfolio_candidates).sort_values(by='User_Pref_Score', ascending=False).reset_index(drop=True)
    
    # 🚨 新增：提取最終名單對應的「正規化特徵矩陣」並存檔
    final_tickers = final_df['ETF'].tolist()
    final_scaled_df = df_scaled[df_scaled['ETF'].isin(final_tickers)].reset_index(drop=True)
    
    final_df.to_csv("csv\\stage2_final_user_universe.csv", index=False)
    final_scaled_df.to_csv("csv\\stage2_normalized_features.csv", index=False)
    
    print("\n✅ 資料已輸出至 csv\\stage2_final_user_universe.csv")
    print("✅ 正規化特徵矩陣已輸出至 csv\\stage2_normalized_features.csv")

# 執行
run_stage2_5_preference_deduplication_yq()

啟動 Stage 2.5: 偏好驅動去重與分群 (白名單過濾與原始數據重構)...
📥 成功載入 24 檔候選 ETF 之原始特徵。

啟動本地快取引擎 (請求總數: 24 檔)...
⚡ 所有請求的 ETF 皆已在本地快取中，直接載入！

=== 🎯 Stage 2.5 群集去重結果 ===
  > 群集 ['SCHX', 'VOO', 'IVV', 'VTI', 'ITOT', 'SCHB', 'SPY', 'VV'] -> 🏆 勝出: VOO (Score: 0.8355)
 ETF  User_Pref_Score
 VOO         0.835545
 IVV         0.824457
 SPY         0.740867
 VTI         0.825085
ITOT         0.807628
SCHX         0.786090
  VV         0.779557
SCHB         0.796025

✅ 資料已輸出至 csv\stage2_final_user_universe.csv
✅ 正規化特徵矩陣已輸出至 csv\stage2_normalized_features.csv
